In [1]:

import os, sys, shutil
import numpy as np
import pandas as sp
import pandas as pd
import nibabel as nib
import json

In [2]:
home = os.path.expanduser("~")
print(home)
atlas_dir = os.path.join(home, 'research_projects/snaplab_tools/data/atlases')

/home/lindenmp


In [3]:
def combine_parcs(parc_x, parc_y):
    index_offset = parc_x.max()  # get parc offset from parc_x
    parc_y_mask = parc_y > 0  # create binary mask of parc_y
    parc_x[parc_y_mask] = 0  # prep parc_x data by zeroing out parc_y voxels
    parc_y += index_offset  # offset parc_y data
    parc_y[~parc_y_mask] = 0  # re-zero voxels in parc_y
    parc_out = parc_x + parc_y  # add parcellations together
    
    return parc_out

In [4]:
msa_max_indices = [16, 32, 50, 54]

for tian_scale in [1, 2, 3, 4]:
    out_dir_msa = os.path.join(atlas_dir, 'GlasserMSA', 'atlas-GlasserMSA{0}'.format(tian_scale))
    if os.path.isdir(out_dir_msa):
        shutil.rmtree(out_dir_msa)
    os.makedirs(out_dir_msa)

    out_dir_msamdtb = os.path.join(atlas_dir, 'GlasserMSAMDTB10', 'atlas-GlasserMSA{0}MDTB10'.format(tian_scale))
    if os.path.isdir(out_dir_msamdtb):
        shutil.rmtree(out_dir_msamdtb)
    os.makedirs(out_dir_msamdtb)

    for space in ['MNI152NLin2009cAsym', 'MNI152NLin6Asym']:
        for res in [1, 2]:
            print(tian_scale, space, res)
            # load parc file
            glasser_file = os.path.join(atlas_dir, 'Glasser', 'atlas-Glasser',
                                        'atlas-Glasser_space-{0}_res-{1}_dseg.nii.gz'.format(space, res))
            glasser = nib.load(glasser_file)
            glasser_data = glasser.get_fdata()
            if glasser_data.ndim == 4:
                glasser_data = np.squeeze(glasser_data)
            
            msa_file = os.path.join(atlas_dir, 'MSA', 'atlas-MSA{0}'.format(tian_scale),
                                    'atlas-MSA{0}_space-{1}_res-{2}_dseg.nii.gz'.format(tian_scale, space, res))
            msa = nib.load(msa_file)
            msa_data = msa.get_fdata()
            
            cere_file = os.path.join(atlas_dir, 'MDTB10',
                                     'atlas-MDTB10_space-{0}_res-{1}_dseg.nii.gz'.format(space, res))
            cere = nib.load(cere_file)
            cere_data = cere.get_fdata()

            # save out glasser + msa
            glasser_msa = combine_parcs(glasser_data, msa_data)
            # save out (overwrite)
            parc_out = nib.Nifti1Image(glasser_msa, affine=glasser.affine, header=glasser.header)
            parc_file_out = os.path.join(out_dir_msa, 'atlas-GlasserMSA{0}_space-{1}_res-{2}_dseg.nii.gz'.format(tian_scale, space, res))
            nib.save(parc_out, parc_file_out)

            json_file = os.path.join(out_dir_msa, 'atlas-GlasserMSA{0}_dseg.json'.format(tian_scale))
            data = {"BIDSVersion": "1.8.0", "Name": "GlasserMSA{0}".format(tian_scale)}  # , "DatasetType": "atlas"
            # creating a JSON string
            json_string = json.dumps(data)
            # storing it in a file
            with open(json_file, "w") as json_data:
                json.dump(data, json_data)
                
            # save out glasser + msa + mdtb
            glasser_msa_cere = combine_parcs(glasser_msa, cere_data)
            # save out (overwrite)
            parc_out = nib.Nifti1Image(glasser_msa_cere, affine=glasser.affine, header=glasser.header)
            parc_file_out = os.path.join(out_dir_msamdtb, 'atlas-GlasserMSA{0}MDTB10_space-{1}_res-{2}_dseg.nii.gz'.format(tian_scale, space, res))
            nib.save(parc_out, parc_file_out)

            json_file = os.path.join(out_dir_msamdtb, 'atlas-GlasserMSA{0}MDTB10_dseg.json'.format(tian_scale))
            data = {"BIDSVersion": "1.8.0", "Name": "GlasserMSA{0}MDTB10".format(tian_scale)}  # , "DatasetType": "atlas"
            # creating a JSON string
            json_string = json.dumps(data)
            # storing it in a file
            with open(json_file, "w") as json_data:
                json.dump(data, json_data)
        
    # update tsv file
    tsv_file = os.path.join(atlas_dir, 'Glasser', 'atlas-Glasser',
                            'atlas-Glasser_dseg.tsv'.format(tian_scale))
    df = pd.read_csv(tsv_file, header=0, index_col=0, sep='\t')

    tsv_file_msa = os.path.join(atlas_dir, 'MSA',
                                'atlas-MSA{0}'.format(tian_scale), 'atlas-MSA{0}_dseg.tsv'.format(tian_scale))
    df_msa = pd.read_csv(tsv_file_msa, header=0, index_col=0, sep='\t')    
    tsv_file_cere = os.path.join(atlas_dir, 'MDTB10',
                                 'atlas-MDTB10_dseg.tsv')
    df_cere = pd.read_csv(tsv_file_cere, header=0, index_col=0, sep='\t')
    
    # save out glasser + msa
    df['cortex'] = True
    df['subcortex'] = False
    df_msa['cortex'] = False
    df_msa['subcortex'] = True
    index_offset = 360
    df_msa.index = (df_msa.index.values + index_offset).astype(int)
    df_msa.index.name = 'index'

    df_out = pd.concat((df, df_msa), axis=0)
    tsv_file_out = os.path.join(out_dir_msa, 'atlas-GlasserMSA{0}_dseg.tsv'.format(tian_scale))
    df_out.to_csv(tsv_file_out, sep="\t")

    # save out glasser + msa + mdtb
    df['cerebellum'] = False
    df_msa['cerebellum'] = False
    df_cere['cortex'] = False
    df_cere['subcortex'] = False
    df_cere['cerebellum'] = True
    df_cere.drop(columns=['color'], inplace=True)
    df_cere['label'] = 'cerebellum_' + df_cere['label'].astype(str)
    df_cere.index = (df_cere.index.values + index_offset + msa_max_indices[tian_scale - 1]).astype(int)
    df_cere.index.name = 'index'

    df_out = pd.concat((df, df_msa, df_cere), axis=0)
    tsv_file_out = os.path.join(out_dir_msamdtb, 'atlas-GlasserMSA{0}MDTB10_dseg.tsv'.format(tian_scale))
    df_out.to_csv(tsv_file_out, sep="\t")

1 MNI152NLin2009cAsym 1
1 MNI152NLin2009cAsym 2
1 MNI152NLin6Asym 1
1 MNI152NLin6Asym 2
2 MNI152NLin2009cAsym 1
2 MNI152NLin2009cAsym 2
2 MNI152NLin6Asym 1
2 MNI152NLin6Asym 2
3 MNI152NLin2009cAsym 1
3 MNI152NLin2009cAsym 2
3 MNI152NLin6Asym 1
3 MNI152NLin6Asym 2
4 MNI152NLin2009cAsym 1
4 MNI152NLin2009cAsym 2
4 MNI152NLin6Asym 1
4 MNI152NLin6Asym 2


## Copy dataset_description

In [5]:
in_file = os.path.join(atlas_dir, 'dataset_description.json')

out_file = os.path.join(atlas_dir, 'GlasserMSA', 'dataset_description.json')
shutil.copyfile(in_file, out_file)

out_file = os.path.join(atlas_dir, 'GlasserMSAMDTB10', 'dataset_description.json')
shutil.copyfile(in_file, out_file)

'/home/lindenmp/research_projects/snaplab_tools/data/atlases/GlasserMSAMDTB10/dataset_description.json'